# External evaluation: BERT-only vs BERT + regex

Compares model performance on `data/external_updated.xlsx` before and after the NFLIS regex pass.
Run top to bottom with the kernel working directory set to `pipeline/`.

- **Before** = BERT predictions thresholded with `best_thresholds.json` (same as `evaluate.py`, source of the old 0.965 Any Drugs accuracy).
- **After** = the same predictions passed through `apply_regex_classifier` (the production BERT+regex pipeline).

In [ ]:
import json
from pathlib import Path

import pandas as pd
import torch
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from classify import TextDataset, create_text_col, apply_regex_classifier

ROOT = Path.cwd().parent
MODEL_DIR = ROOT / "models" / "bert_models" / "bioclinicalbert"

DRUG_COLS = [
    "Methamphetamine", "Heroin", "Cocaine", "Fentanyl", "Alcohol",
    "Prescription.opioids", "Any Opioids", "Benzodiazepines", "Others", "Any Drugs",
]

## 1. Load data and true labels

`external_updated.xlsx` uses different column names than the model, so rename:
`Any opioid` → `Any Opioids`, `Prescription opioids` → `Prescription.opioids`.

It has no `Any Drugs` truth column, so that one is pulled from `external_test.csv`
(same 3335 cases) by joining on `Case.Number`.

In [ ]:
external_df = pd.read_csv(ROOT / "data" / "recoded_ext_test2801_recode.csv")
external_df = external_df.rename(columns={
    "Any opioid": "Any Opioids",
    "Prescription opioids": "Prescription.opioids",
})

external_df["Any Drugs"] = external_df["Any Drugs"].fillna(0).astype(int)

y_true = external_df[DRUG_COLS].astype(int).reset_index(drop=True)
y_true.sum()

In [ ]:
external_df['text']

## 2. Build the model input text

The model expects a `text` column built from the cause fields. The external set only has
`Combined_text`, so rename it to `CauseA` — `create_text_col` then builds `text` from it,
and the regex classifier searches `CauseA` too.

In [ ]:
feats = external_df.drop(columns=DRUG_COLS).rename(columns={"text": "CauseA"})
feats = create_text_col(feats).reset_index(drop=True)
feats[["Case.Number", "text"]].head()

## 3. BERT predicted probabilities

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_DIR, num_labels=len(DRUG_COLS), problem_type="multi_label_classification"
).to(device)
model.eval()

loader = DataLoader(TextDataset(feats["text"].tolist(), tokenizer), batch_size=32)
probs = []
with torch.no_grad():
    for batch in tqdm(loader):
        inputs = {k: v.to(device) for k, v in batch.items()}
        probs.append(torch.sigmoid(model(**inputs).logits).cpu())
probs = torch.cat(probs).numpy()
probs.shape

## 4. "Before" predictions: per-label best thresholds

Same thresholding as `evaluate.py` — this is the arm comparable to the old 0.965 baseline.

In [ ]:
with open(MODEL_DIR / "best_thresholds.json") as f:
    thresholds = json.load(f)

before = pd.DataFrame(
    {col: (probs[:, i] >= thresholds[col]).astype(int) for i, col in enumerate(DRUG_COLS)}
)
before.sum()

## 5. \"After\" predictions: apply the regex pass on top of BERT

In [ ]:
pred_df = pd.concat([feats, before], axis=1)
after_df = apply_regex_classifier(pred_df.copy())
after = after_df[DRUG_COLS].astype(int)

## 6. Per-class metrics, before vs after

In [ ]:
def per_class_metrics(y, p):
    return pd.DataFrame({
        col: {
            "accuracy": accuracy_score(y[col], p[col]),
            "precision": precision_score(y[col], p[col], zero_division=0),
            "recall": recall_score(y[col], p[col], zero_division=0),
            "f1": f1_score(y[col], p[col], zero_division=0),
        }
        for col in DRUG_COLS
    }).T

m_before = per_class_metrics(y_true, before)
m_after = per_class_metrics(y_true, after)

print("=== BERT only (before regex) ===")
display(m_before.round(4))
print("=== BERT + regex (after) ===")
display(m_after.round(4))

In [ ]:
comparison = m_before.join(m_after, lsuffix="_before", rsuffix="_after")
comparison["accuracy_delta"] = comparison["accuracy_after"] - comparison["accuracy_before"]
comparison["f1_delta"] = comparison["f1_after"] - comparison["f1_before"]
comparison.to_csv(ROOT / "reports" / "eval_external_regex_comparison.csv")
comparison[["accuracy_before", "accuracy_after", "accuracy_delta", "f1_before", "f1_after", "f1_delta"]].round(4)

## 7. Inspect rows the regex changed against the truth labels

Shows, per class, the rows regex flipped 0→1 where the truth label is 0 — the source of any
precision drop. For `Prescription.opioids` these are mostly labeling-convention mismatches
(generic "opioid(s)", nitazenes, mitragynine/kratom) plus the "dextro/levo methorphan" pattern.

In [ ]:
from regex_classifier import build_patterns, DEFAULT_NFLIS

patterns = build_patterns(DEFAULT_NFLIS)
col = "Prescription.opioids"

added_fp = (before[col] == 0) & (after[col] == 1) & (y_true[col] == 0)
print(f"{col}: {int(added_fp.sum())} regex-added rows disagree with truth labels")

rows = []
for i in feats.index[added_fp]:
    m = patterns[col].search(feats.at[i, "text"])
    rows.append({
        "Case.Number": feats.at[i, "Case.Number"],
        "matched_term": m.group() if m else None,
        "Prescription.opioids_true": y_true.at[i, "Prescription.opioids"],
        "text": feats.at[i, "text"][:120],
        "Prescription.opioids": after.at[i, "Prescription.opioids"]
    })
pd.DataFrame(rows)

In [ ]:
prescription_opioids = pd.DataFrame(rows)

In [ ]:
prescription_opioids.to_csv("prescopioidsmistmatch.csv")

In [ ]:
external_df[external_df['Case.Number'] == '2023-13916']['Any Opioids']

In [ ]:
external_df[external_df['Case.Number'] == '2023-13916']['text']